<a href="https://colab.research.google.com/github/FelipeBlancoP/Laboratorio-1---Clasificador-de-Textos/blob/ramaMau/tf_idf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1.- Importar Dataset

In [1]:
import os
from sklearn.datasets import fetch_20newsgroups

categories = ["comp.graphics", "sci.space", "rec.sport.hockey", "talk.politics.mideast"]

output_dir = "dataset"
os.makedirs(output_dir, exist_ok=True)

dataset = fetch_20newsgroups(subset="train", categories=categories, shuffle=True, random_state=42)

# Limitar a 6 documentos por categoría
docs_per_category = {cat: 0 for cat in categories}
for i, text in enumerate(dataset.data):
    label = dataset.target_names[dataset.target[i]]
    if docs_per_category[label] >= 6:
        continue

    folder = os.path.join(output_dir, label)
    os.makedirs(folder, exist_ok=True)

    filename = os.path.join(folder, f"doc_{docs_per_category[label]}.txt")
    with open(filename, "w", encoding="utf-8", errors="ignore") as f:
        f.write(text)

    docs_per_category[label] += 1

    if all(v >= 6 for v in docs_per_category.values()):
        break

2.- Limpieza de Dataset


  a.- Limpieza: Pasar todo a minúsculas, quitar caracteres especiales, tildes.

In [3]:
import re
import unicodedata

def limpiar_texto(texto):
    # 1. Pasar a minúsculas
    texto = texto.lower()

    # 2. Quitar tildes (acentos)
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8', 'ignore')

    # 3. Eliminar caracteres especiales, números y signos de puntuación
    texto = re.sub(r'[^a-z\s]', ' ', texto)

    # 4. Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto



base_dir = "dataset"

for categoria in os.listdir(base_dir):
    carpeta = os.path.join(base_dir, categoria)
    if not os.path.isdir(carpeta):
        continue

    print(f"Procesando categoría: {categoria}")

    for archivo in os.listdir(carpeta):
        ruta = os.path.join(carpeta, archivo)
        with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
            texto = f.read()

        texto_limpio = limpiar_texto(texto)

        # Si quieres, puedes guardar el texto limpio en una carpeta aparte:
        carpeta_salida = os.path.join("dataset_limpio", categoria)
        os.makedirs(carpeta_salida, exist_ok=True)
        with open(os.path.join(carpeta_salida, archivo), "w", encoding="utf-8") as f:
            f.write(texto_limpio)

Procesando categoría: comp.graphics
Procesando categoría: sci.space
Procesando categoría: talk.politics.mideast
Procesando categoría: rec.sport.hockey


b.- Tokenizar

In [7]:
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
from nltk.tokenize import word_tokenize
import os

# Carpeta donde están tus textos limpios
base_dir = "dataset_limpio"

def tokenizar_texto(texto):
    # Tokeniza el texto (separa en palabras)
    return word_tokenize(texto)

for categoria in os.listdir(base_dir):
    carpeta = os.path.join(base_dir, categoria)
    if not os.path.isdir(carpeta):
        continue

    print(f"Tokenizando categoría: {categoria}")

    for archivo in os.listdir(carpeta):
        ruta = os.path.join(carpeta, archivo)
        with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
            texto = f.read()

        tokens = tokenizar_texto(texto)

        print(f"{archivo}: {len(tokens)} tokens")


Tokenizando categoría: comp.graphics
doc_4.txt: 94 tokens
doc_2.txt: 141 tokens
doc_0.txt: 165 tokens
doc_3.txt: 182 tokens
doc_5.txt: 309 tokens
doc_1.txt: 124 tokens
Tokenizando categoría: sci.space
doc_4.txt: 149 tokens
doc_2.txt: 139 tokens
doc_0.txt: 290 tokens
doc_3.txt: 213 tokens
doc_5.txt: 151 tokens
doc_1.txt: 124 tokens
Tokenizando categoría: talk.politics.mideast
doc_4.txt: 228 tokens
doc_2.txt: 587 tokens
doc_0.txt: 260 tokens
doc_3.txt: 156 tokens
doc_5.txt: 177 tokens
doc_1.txt: 259 tokens
Tokenizando categoría: rec.sport.hockey
doc_4.txt: 108 tokens
doc_2.txt: 199 tokens
doc_0.txt: 129 tokens
doc_3.txt: 301 tokens
doc_5.txt: 183 tokens
doc_1.txt: 177 tokens


c.- Eliminar stopwords

In [9]:
import os
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Usamos inglés porque los textos del dataset 20newsgroups están en inglés
stop_words = set(stopwords.words('english'))

def eliminar_stopwords(tokens):
    """
    Recibe una lista de tokens y devuelve otra sin stopwords ni palabras muy cortas.
    """
    tokens_filtrados = [t for t in tokens if t.lower() not in stop_words and len(t) > 1]
    return tokens_filtrados


input_dir = "dataset_limpio"
output_dir = "dataset_tokens"
os.makedirs(output_dir, exist_ok=True)

for categoria in os.listdir(input_dir):
    carpeta = os.path.join(input_dir, categoria)
    if not os.path.isdir(carpeta):
        continue

    carpeta_salida = os.path.join(output_dir, categoria)
    os.makedirs(carpeta_salida, exist_ok=True)

    print(f"Procesando categoría: {categoria}")

    for archivo in os.listdir(carpeta):
        ruta = os.path.join(carpeta, archivo)
        with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
            texto = f.read()

        # Tokenizar
        tokens = word_tokenize(texto)

        # Eliminar stopwords
        tokens_filtrados = eliminar_stopwords(tokens)

        # Guardar resultado
        with open(os.path.join(carpeta_salida, archivo), "w", encoding="utf-8") as f:
            f.write(" ".join(tokens_filtrados))

print("Stopwords eliminadas y textos guardados en 'dataset_tokens/'")

Procesando categoría: comp.graphics
Procesando categoría: sci.space
Procesando categoría: talk.politics.mideast
Procesando categoría: rec.sport.hockey
✅ Stopwords eliminadas y textos guardados en 'dataset_tokens/'
